# 📘 Week 3 · Day 20-21: 하이퍼파라미터 튜닝 (Optuna) + 앙상블 & 스태킹

## 🎯 학습 목표
- **Optuna**로 효율적 하이퍼파라미터 탐색
- **Grid / Random / Bayesian Search** 차이 이해
- **Voting / Averaging / Weighted / Rank Averaging** 앙상블
- **Stacking / Blending** 구조와 직접 구현
- 실전 Kaggle 상위권 파이프라인 구조


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.datasets import make_classification
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings('ignore')
np.random.seed(0)

---

## 1. 하이퍼파라미터 탐색 방법

| 방법 | 특징 | 언제 |
|------|------|------|
| **Grid Search** | 격자로 전수 탐색 | 파라미터 3개 이하, 공간 좁을 때 |
| **Random Search** | 무작위 샘플 | 파라미터 많을 때 (Grid보다 효율) |
| **Bayesian (Optuna)** | 이전 결과로 다음 시도 추천 | **Kaggle 표준** |
| **Population-based** | 유전 알고리즘 | 특수한 경우 |

### 왜 Random Search가 Grid보다 좋은가?
고차원에서 **중요한 파라미터는 몇 개뿐**인데, Grid는 중요하지 않은 파라미터에 자원을 낭비함.


---

## 2. Grid Search 예시 (참고용)

In [ ]:
from sklearn.model_selection import GridSearchCV
import lightgbm as lgb

X, y = make_classification(n_samples=2000, n_features=15, random_state=0)

param_grid = {
    'num_leaves':    [15, 31, 63],
    'learning_rate': [0.05, 0.1],
    'max_depth':     [4, 6, -1],
}

gs = GridSearchCV(
    lgb.LGBMClassifier(n_estimators=100, random_state=0, verbose=-1, n_jobs=-1),
    param_grid, cv=3, scoring='roc_auc', n_jobs=-1
)
gs.fit(X, y)
print(f"Best AUC: {gs.best_score_:.4f}")
print(f"Best params: {gs.best_params_}")

---

## 3. Optuna — Bayesian 최적화의 표준

### 철학
1. 탐색 공간을 **함수(objective)** 로 정의
2. Optuna가 **이전 trial 결과를 학습**해서 다음 파라미터 추천
3. **Pruning**: 성능 낮은 trial 조기 중단

### 설치
```bash
pip install optuna
```


In [ ]:
import optuna
from sklearn.model_selection import StratifiedKFold

def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves':       trial.suggest_int('num_leaves', 15, 255),
        'max_depth':        trial.suggest_int('max_depth', 3, 12),
        'min_child_samples':trial.suggest_int('min_child_samples', 5, 100),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 0, 'verbose': -1, 'n_jobs': -1
    }

    skf = StratifiedKFold(5, shuffle=True, random_state=0)
    scores = []
    for tr, val in skf.split(X, y):
        m = lgb.LGBMClassifier(**params)
        m.fit(X[tr], y[tr],
              eval_set=[(X[val], y[val])],
              callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
        scores.append(roc_auc_score(y[val], m.predict_proba(X[val])[:, 1]))
    return np.mean(scores)

# 탐색 시작 (시간상 20회만)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=False)

print(f"Best AUC: {study.best_value:.4f}")
print(f"Best params:\n{study.best_params}")

### 3.1 Optuna 시각화

In [ ]:
# 최적화 히스토리
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

trials = study.trials_dataframe()
axes[0].plot(trials['number'], trials['value'], 'o-', alpha=0.6)
axes[0].axhline(study.best_value, c='red', ls='--', label=f'best={study.best_value:.4f}')
axes[0].set_xlabel('trial'); axes[0].set_ylabel('AUC')
axes[0].set_title('Optuna Optimization History'); axes[0].legend(); axes[0].grid(True)

# 파라미터 중요도
try:
    importances = optuna.importance.get_param_importances(study)
    pd.Series(importances).plot(kind='barh', ax=axes[1], color='coral')
    axes[1].set_title('Hyperparameter Importance')
except Exception:
    axes[1].text(0.5, 0.5, 'more trials needed', ha='center')
plt.tight_layout(); plt.show()

### 3.2 Pruning (조기 중단)로 시간 절약

In [ ]:
from optuna.integration import LightGBMPruningCallback

def objective_with_pruning(trial):
    params = {
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves':       trial.suggest_int('num_leaves', 15, 255),
        'random_state': 0, 'verbose': -1, 'n_jobs': -1, 'metric': 'auc'
    }
    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=0)
    m = lgb.LGBMClassifier(n_estimators=300, **params)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
          callbacks=[
              lgb.early_stopping(30),
              lgb.log_evaluation(0),
              LightGBMPruningCallback(trial, 'auc', valid_name='valid_0')
          ])
    return roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])

study2 = optuna.create_study(direction='maximize',
                             pruner=optuna.pruners.MedianPruner(n_warmup_steps=30))
study2.optimize(objective_with_pruning, n_trials=15, show_progress_bar=False)
print(f"Pruning 적용 Best AUC: {study2.best_value:.4f}")

---

## 4. 앙상블 (Ensemble)

### 왜 앙상블하는가?
**서로 다른 실수를 하는 모델들**을 합치면 실수가 상쇄됨.

### 4.1 Voting (분류)
- **Hard**: 다수결
- **Soft**: 확률 평균 (더 좋음)

### 4.2 Averaging (회귀/확률)
- **Simple Average**: `(pred1 + pred2 + pred3) / 3`
- **Weighted Average**: CV 점수 기반 가중치
- **Rank Averaging**: 각 모델의 rank를 평균 (지표가 AUC일 때 좋음)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

# 3개 모델 학습
m1 = LogisticRegression(max_iter=1000, random_state=0).fit(X_tr, y_tr)
m2 = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
m3 = lgb.LGBMClassifier(n_estimators=200, random_state=0, verbose=-1, n_jobs=-1).fit(X_tr, y_tr)

p1 = m1.predict_proba(X_te)[:, 1]
p2 = m2.predict_proba(X_te)[:, 1]
p3 = m3.predict_proba(X_te)[:, 1]

print("개별 AUC:")
print(f"  LogReg : {roc_auc_score(y_te, p1):.4f}")
print(f"  RF     : {roc_auc_score(y_te, p2):.4f}")
print(f"  LGBM   : {roc_auc_score(y_te, p3):.4f}")

# Simple Average
p_avg = (p1 + p2 + p3) / 3
print(f"\nSimple Avg AUC: {roc_auc_score(y_te, p_avg):.4f}")

# Weighted Average (LGBM > RF > LogReg)
p_wavg = 0.2*p1 + 0.3*p2 + 0.5*p3
print(f"Weighted  AUC: {roc_auc_score(y_te, p_wavg):.4f}")

# Rank Average (AUC 지표에 특히 유용)
from scipy.stats import rankdata
p_rank = (rankdata(p1) + rankdata(p2) + rankdata(p3)) / 3
print(f"Rank Avg  AUC: {roc_auc_score(y_te, p_rank):.4f}")

### 앙상블이 잘 되려면?

1. **모델이 서로 다른 실수**를 해야 함
   - → 모델의 예측 간 **상관관계가 낮아야** 함
2. **개별 성능이 어느 정도** 되어야 함


In [ ]:
# 모델 간 예측 상관관계
preds = pd.DataFrame({'LogReg': p1, 'RF': p2, 'LGBM': p3})
print("예측 상관관계:")
print(preds.corr().round(3))
# 상관 0.95 이상이면 앙상블 효과 적음

---

## 5. Stacking — Kaggle 상위권의 정석

### 구조
```
Level 0 (Base models): 여러 다양한 모델들이 OOF 예측 생성
Level 1 (Meta model): OOF 예측을 입력으로 받아 최종 예측
```

### 핵심 개념
- **OOF (Out-of-Fold) 예측**: KFold로 학습하면서 val fold 예측만 모음
- → **Leakage 없이** base 모델 예측을 Meta 모델의 피처로 사용


In [ ]:
def get_oof(model, X, y, X_test, n_splits=5, random_state=0):
    '''OOF 예측 + test 평균 예측 반환'''
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    skf = StratifiedKFold(n_splits, shuffle=True, random_state=random_state)

    for tr, val in skf.split(X, y):
        m = type(model)(**model.get_params())
        m.fit(X[tr], y[tr])
        oof[val] = m.predict_proba(X[val])[:, 1]
        test_pred += m.predict_proba(X_test)[:, 1] / n_splits
    return oof, test_pred

# Level 0 모델들
models = {
    'logreg': LogisticRegression(max_iter=1000, random_state=0),
    'rf':     RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1),
    'lgbm':   lgb.LGBMClassifier(n_estimators=200, random_state=0, verbose=-1, n_jobs=-1),
}

oof_preds = {}
test_preds = {}
for name, m in models.items():
    oof_preds[name], test_preds[name] = get_oof(m, X_tr, y_tr, X_te, n_splits=5)
    print(f"{name:7s} OOF AUC: {roc_auc_score(y_tr, oof_preds[name]):.4f}")

# Level 0 예측을 컬럼으로 쌓기
X_tr_stack = np.column_stack(list(oof_preds.values()))
X_te_stack = np.column_stack(list(test_preds.values()))
print(f"\nLevel 1 train shape: {X_tr_stack.shape}")

In [ ]:
# Level 1 Meta model (간단한 LogReg 추천)
meta = LogisticRegression(max_iter=1000, C=1.0)
meta.fit(X_tr_stack, y_tr)
stack_pred = meta.predict_proba(X_te_stack)[:, 1]

print(f"Stacking AUC: {roc_auc_score(y_te, stack_pred):.4f}")
print(f"Meta coefs  : {dict(zip(models.keys(), meta.coef_[0].round(3)))}")

### 5.1 Blending — Stacking의 단순 버전

- Stacking: KFold 기반 OOF 사용 (견고)
- Blending: 단순 holdout val set 사용 (간단, 빠름, 데이터 적게 사용)

### 5.2 Stacking 설계 팁
1. **Level 0은 다양성**: 서로 다른 종류 (LGBM, XGB, CatBoost, NN, KNN...)
2. **Level 1은 단순**: Ridge / LogReg 권장 (과적합 방지)
3. **2단 이상은 주의**: 오버피팅 쉬움
4. **seed ensemble**: 같은 모델 다른 seed로 여러 번 돌려 평균


---

## 6. Seed Ensemble — 간단하지만 강력

In [ ]:
# 같은 LightGBM을 다른 random_state로 5번 학습
seed_preds = []
for seed in [0, 1, 2, 3, 4]:
    m = lgb.LGBMClassifier(n_estimators=200, random_state=seed, verbose=-1, n_jobs=-1)
    m.fit(X_tr, y_tr)
    seed_preds.append(m.predict_proba(X_te)[:, 1])

# 평균
seed_avg = np.mean(seed_preds, axis=0)
print(f"단일 seed AUC: {roc_auc_score(y_te, seed_preds[0]):.4f}")
print(f"5-seed avg AUC: {roc_auc_score(y_te, seed_avg):.4f}")
# 보통 작은 차이지만 안정적인 상승

---

## 7. 실전 Kaggle Top-10 파이프라인 구조

```
┌────────────────────────────────────────────────────┐
│         Feature Engineering (공통)                  │
└────────────────────────────────────────────────────┘
              │
    ┌─────────┼─────────┬─────────┬─────────┐
    ▼         ▼         ▼         ▼         ▼
 LightGBM   XGBoost  CatBoost   NN       Linear
 (seed×5)   (seed×5) (seed×5)  (fold×5) (Ridge)
    │         │         │         │         │
    └─────────┴─────────┴─────────┴─────────┘
              │     Level 0 OOF
              ▼
        ┌──────────┐
        │  Meta     │  ← Ridge / LogReg / LGBM
        └──────────┘
              │
              ▼
         Final pred
```


---

## 📝 Day 20-21 체크리스트
- [ ] Grid vs Random vs Bayesian 차이
- [ ] Optuna objective 함수 작성 가능
- [ ] Pruning 적용
- [ ] Simple/Weighted/Rank averaging 구현
- [ ] OOF + Stacking 직접 구현
- [ ] Seed ensemble 적용

다음 노트북에서 **실전 Titanic / House Prices 대회**를 처음부터 끝까지 풀어봅니다.